#Statement of Intent
My goal is to build a full ETL pipepline and model for tennis analytics. The model will predict the winner of a match given certain parameters.

#Data Sourcing
The data is sourced from match statistics provided by Jeff Sackmann.

The last 4 years worth of matches from Jeff's GitHub repo have been stored in s3 on a free AWS account. I am starting the feature engineering by initiating the spark session and pulling the raw files from the s3 bucket.

In [0]:
from pyspark.sql import (
    SparkSession,
    types,
    functions as F,
)

from pyspark.sql.window import Window

spark = SparkSession.builder.appName("TennisAnalytics").getOrCreate()

I convert the files to delta for faster pulling, version control, and updating records if needed. I only need to run that cell once.

In [0]:
#Convert to delta
df = spark.read.csv("s3://data-storage-for-projects/Tennis Analytics Project/raw/"
                    , header=True, inferSchema=True)
df.write.format("delta").mode("overwrite").save("s3://data-storage-for-projects/Tennis Analytics Project/delta/")

In [0]:
#Bring data into notebook
df = spark.read.format("delta").load("s3://data-storage-for-projects/Tennis Analytics Project/delta/")
display(df)

#Baseline Model aka V1
For my baseline models, I am mostly using tournament conditions such as surface, tournament level, best of, and round of the match. I will also include rankings of each player, but I will need to have two rows per match so that each player is player 1. The data in its current form has winners all in one column and could train the model incorrectly. As for the types of models, I plan to create and compare three--logisitic regression, random forest, and gradient boosting.

In [0]:
#Columns for player a and player b rank, age, and hand. It also says if a won. Two are used to prevent data leakage.
d1 = df.select('surface', 'tourney_level', 'round',
        'best_of', 'winner_hand', 'loser_hand',
        F.col('winner_id').alias('id_a'), F.col('loser_id').alias('id_b'), 
        F.col('winner_rank').alias('id_a_rank'), F.col('loser_rank').alias('id_b_rank'),
        F.col('winner_age').alias('id_a_age'), F.col('loser_age').alias('id_b_age')
        ).withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
        .withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
        .withColumn('id_a_hand', F.when(F.col('winner_hand')=='R', 0).otherwise(1)) \
        .withColumn('id_b_hand', F.when(F.col('loser_hand')=='R', 0).otherwise(1))  \
        .withColumn("round_encoded",
        F.when(df.round == "RR", 0)
        .when(df.round == "R128", 1)
        .when(df.round == "R64", 2)
        .when(df.round == "R32", 3)
        .when(df.round == "R16", 4)
        .when(df.round == "QF", 5)
        .when(df.round == "SF", 6)
        .when(df.round == "BR", 6.5)
        .when(df.round == "F", 7)
        .otherwise(None)
        ) \
        .withColumn('id_a_won', F.lit(1))


d2 = df.select('surface', 'tourney_level', 'round',
        'best_of', 'winner_hand', 'loser_hand',
        F.col('loser_id').alias('id_a'), F.col('winner_id').alias('id_b'), 
        F.col('loser_rank').alias('id_a_rank'), F.col('winner_rank').alias('id_b_rank'),
        F.col('loser_age').alias('id_a_age'), F.col('winner_age').alias('id_b_age')
        ).withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
        .withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
        .withColumn('id_a_hand', F.when(F.col('loser_hand')=='R', 0).otherwise(1)) \
        .withColumn('id_b_hand', F.when(F.col('winner_hand')=='R', 0).otherwise(1)) \
        .withColumn("round_encoded",
        F.when(F.col('round') == "RR", 0)
        .when(F.col('round') == "R128", 1)
        .when(F.col('round') == "R64", 2)
        .when(F.col('round') == "R32", 3)
        .when(F.col('round') == "R16", 4)
        .when(F.col('round') == "QF", 5)
        .when(F.col('round') == "SF", 6)
        .when(F.col('round') == "BR", 6)
        .when(F.col('round') == "F", 7)
        .otherwise(None)
        ) \
        .withColumn('id_a_won', F.lit(0))

d = d1.union(d2).drop('winner_hand', 'loser_hand', 'round') \
        .filter(F.col('surface').isNotNull() & F.col('rank_diff').isNotNull()
                & F.col('id_a_age').isNotNull() & F.col('id_b_age').isNotNull()) 
display(d)

In [0]:
#Create ML Pipeline starting with training data, string indexer and one hot encoder for logisitic regression
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

#Traininng and testing data
train_df, test_df = d.randomSplit([.8, .2], seed=42)

#Indexing and Encoding
indexer_surface = StringIndexer(inputCol='surface', outputCol='surface_index')
encoder_surface = OneHotEncoder(inputCol='surface_index', outputCol='surface_vec')

indexer_tourney = StringIndexer(inputCol='tourney_level', outputCol='tourney_index')
encoder_tourney = OneHotEncoder(inputCol='tourney_index', outputCol='tourney_vec')

#X variables
feature_cols = ['surface_vec', 'tourney_vec', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

In [0]:
#Logistic Regression Model
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features', labelCol='id_a_won')

pipeline_lr = Pipeline(stages = [
    indexer_surface, encoder_surface,
    indexer_tourney, encoder_tourney,
    assembler,
    lr
])

lr_model = pipeline_lr.fit(train_df)

lr_predict = lr_model.transform(test_df)

In [0]:
#Random Forest
from pyspark.ml.classification import RandomForestClassifier

# indexer_surface = StringIndexer(inputCol='surface', outputCol='surface_encoded', handleInvalid='keep')
# indexer_tourney = StringIndexer(inputCol='tourney_level', outputCol='tourney_encoded', handleInvalid='keep')

# feature_cols = ['surface_encoded', 'tourney_encoded', 'round_encoded', 'best_of', 'id_a_rank',
#                 'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

# assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

rf = RandomForestClassifier(featuresCol='features', labelCol='id_a_won', numTrees=3, maxDepth=2)

pipeline_rf = Pipeline(stages = [
    indexer_surface,
    indexer_tourney,
    assembler,
    rf
])

rf_model = pipeline_rf.fit(train_df)

rf_predict = rf_model.transform(test_df)